# 🔬 D2Vformer Horizon Generalization Experiment
## *True Horizon Generalization in Cross-Temporal Attention*

**Research Question:** Does the horizon-independent cross-temporal attention formulation of PureD2Vformer provide reliable zero-shot forecasting for unseen prediction horizons?

| Model | Regime | Description |
|---|---|---|
| **PureD2Vformer** | Zero-Shot | Trained ONCE at O=48, tested at all horizons |
| **DLinear** | Retrained | Separately trained per horizon (strong baseline) |
| **Repository-D2Vformer** | Retrained | Official implementation, separately trained per horizon |
| **Persistence** | No-learning | Last observed value repeated (control) |

**Datasets:** ETTh1, Exchange Rate | **Seeds:** 42, 43, 44 | **Horizons:** 24, 48, 96, 192, 336, 720

> ⚠️ **IMPORTANT:** Go to `Runtime → Change runtime type → T4 GPU` before running!


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Setup — Install deps, clone repo for datasets and original model files
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'pandas',
                'numpy', 'matplotlib', 'tqdm'], check=False)

if not os.path.exists('/content/D2Vformer'):
    os.system('git clone --quiet https://github.com/TCCofWANG/D2Vformer.git /content/D2Vformer')
    print('✓ Repo cloned.')
else:
    print('✓ Repo already present.')

for d in ['models', 'baselines', 'myutils', 'results/raw', 'results/checkpoints', 'results/plots']:
    os.makedirs(f'/content/D2Vformer/{d}', exist_ok=True)

# Ensure exchange_rate.csv is downloaded and formatted
exch_csv = '/content/D2Vformer/datasets/exchange_rate/exchange_rate.csv'
if not os.path.exists(exch_csv):
    print('Downloading and formatting exchange_rate dataset...')
    import urllib.request, gzip, io, pandas as pd
    url = 'https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/master/exchange_rate/exchange_rate.txt.gz'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = urllib.request.urlopen(req).read()
    f = gzip.GzipFile(fileobj=io.BytesIO(data))
    df = pd.read_csv(f, header=None)
    dates = pd.date_range(start='1990-01-01', periods=len(df), freq='D')
    df.columns = ['0', '1', '2', '3', '4', '5', '6', 'OT']
    df.insert(0, 'date', dates.strftime('%Y-%m-%d'))
    os.makedirs('/content/D2Vformer/datasets/exchange_rate', exist_ok=True)
    df.to_csv(exch_csv, index=False)
    print(f'✓ exchange_rate.csv ready: {df.shape}')
else:
    print('✓ exchange_rate.csv already present.')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU for 10x speedup.')


## Cell 2 — Write all model and utility source files

In [ ]:
# Write all custom source files
import os

# Patch upstream bug: official repo omitted self.trend_linear in model/D2Vformer.py
repo_m = '/content/D2Vformer/model/D2Vformer.py'
if os.path.exists(repo_m):
    with open(repo_m, 'r', encoding='utf-8') as f: content = f.read()
    if 'self.trend_linear' not in content:
        content = content.replace(
            'self.all_linear = nn.Linear(self.seq_len, self.d_model)',
            'self.all_linear = nn.Linear(self.seq_len, self.d_model)\n        self.trend_linear = nn.Linear(self.seq_len, self.d_model)'
        )
        with open(repo_m, 'w', encoding='utf-8') as f: f.write(content)
        print('✓ Patched official repo bug: added self.trend_linear to model/D2Vformer.py')

open('/content/D2Vformer/models/pure_d2vformer.py','w').write("\nimport torch, torch.nn as nn, math, sys\nsys.path.append('/content/D2Vformer')\nfrom layers.Revin import RevIN\n\nclass PureD2Vformer(nn.Module):\n    '''\n    Exact mathematical implementation of D2Vformer (arXiv:2409.11024v1, Eq.1-10).\n    Zero trainable parameters depend on output horizon O. Fully horizon-independent.\n    '''\n    def __init__(self, c_in, seq_len=96, d_model=128, d_ff=256, k_freq=16, dropout=0.05):\n        super().__init__()\n        self.seq_len=seq_len; self.c_in=c_in; self.d_model=d_model; self.k_freq=k_freq\n        self.revin = RevIN(c_in, affine=True, subtract_last=False)\n        self.tfe = nn.Linear(c_in, d_model)\n        self.w_T = nn.Parameter(torch.randn(seq_len))\n        self.b_T = nn.Parameter(torch.zeros(d_model))\n        self.W_S = nn.Parameter(torch.randn(k_freq, seq_len))\n        self.B_S = nn.Parameter(torch.zeros(k_freq, d_model))\n        self.b_1 = nn.Parameter(torch.zeros(d_model,1,1))\n        self.B_2 = nn.Parameter(torch.zeros(k_freq,d_model,1,1))\n        self.b_3 = nn.Parameter(torch.zeros(d_model,1,1))\n        self.B_4 = nn.Parameter(torch.zeros(k_freq,d_model,1,1))\n        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff),nn.GELU(),nn.Dropout(dropout),nn.Linear(d_ff,c_in))\n\n    def forward(self, x_enc, x_mark_enc, y_mark_dec):\n        B,L,D = x_enc.shape; O = y_mark_dec.shape[1]\n        x_norm = self.revin(x_enc, 'norm')\n        T = self.tfe(x_norm)\n        v_T = torch.einsum('l,blh->bh', self.w_T, T) + self.b_T\n        Omega_S = torch.einsum('kl,blh->bkh', self.W_S, T) + self.B_S\n        E_lin = (v_T.unsqueeze(2).unsqueeze(3)*x_mark_enc.unsqueeze(1)+self.b_1).unsqueeze(1)\n        E_har = torch.sin(Omega_S.unsqueeze(3).unsqueeze(4)*x_mark_enc.unsqueeze(1).unsqueeze(2)+self.B_2)\n        D_x = torch.cat([E_lin,E_har],dim=1).mean(dim=-1)\n        F_lin = (v_T.unsqueeze(2).unsqueeze(3)*y_mark_dec.unsqueeze(1)+self.b_3).unsqueeze(1)\n        F_har = torch.sin(Omega_S.unsqueeze(3).unsqueeze(4)*y_mark_dec.unsqueeze(1).unsqueeze(2)+self.B_4)\n        D_y = torch.cat([F_lin,F_har],dim=1).mean(dim=-1)\n        Dx = D_x.permute(0,2,3,1); Dy = D_y.permute(0,2,3,1)\n        A = torch.softmax(torch.einsum('bhok,bhlk->bhol',Dy,Dx)/math.sqrt(self.k_freq+1), dim=-1)\n        Yt = torch.einsum('bhol,bhl->bho', A, T.transpose(1,2))\n        return self.revin(self.ffn(Yt.transpose(1,2)), 'denorm'), A\n")
open('/content/D2Vformer/baselines/dlinear.py','w').write('\nimport torch, torch.nn as nn\nclass moving_avg(nn.Module):\n    def __init__(self,ks,stride):\n        super().__init__(); self.ks=ks; self.avg=nn.AvgPool1d(ks,stride=stride,padding=0)\n    def forward(self,x):\n        f=x[:,0:1,:].repeat(1,(self.ks-1)//2,1); e=x[:,-1:,:].repeat(1,(self.ks-1)//2,1)\n        return self.avg(torch.cat([f,x,e],1).permute(0,2,1)).permute(0,2,1)\nclass series_decomp(nn.Module):\n    def __init__(self,ks): super().__init__(); self.ma=moving_avg(ks,stride=1)\n    def forward(self,x): m=self.ma(x); return x-m,m\nclass DLinear(nn.Module):\n    def __init__(self,seq_len,pred_len,c_in,moving_avg_kernel=25,individual=False):\n        super().__init__(); self.decomp=series_decomp(moving_avg_kernel)\n        self.LS=nn.Linear(seq_len,pred_len); self.LT=nn.Linear(seq_len,pred_len)\n    def forward(self,x,*a,**k):\n        s,t=self.decomp(x); s,t=s.permute(0,2,1),t.permute(0,2,1)\n        return (self.LS(s)+self.LT(t)).permute(0,2,1)\n')
open('/content/D2Vformer/baselines/persistence.py','w').write('\nimport torch, torch.nn as nn\nclass PersistenceBaseline(nn.Module):\n    def __init__(self): super().__init__()\n    def forward(self,x,pred_len,*a,**k): return x[:,-1:,:].repeat(1,pred_len,1)\n')
open('/content/D2Vformer/baselines/repository_d2vformer.py','w').write("\nimport argparse, sys, torch, torch.nn as nn\nsys.path.insert(0,'/content/D2Vformer')\nfrom model.D2Vformer import D2Vformer\ndef create_repository_d2vformer(c_in,seq_len,pred_len,mask_spectrum,d_model=128,\n                                  patch_len=4,stride=2,d_mark=4,T2V_outmodel=16,dropout=0.05):\n    p=argparse.ArgumentParser()\n    for k,v in [('seq_len',seq_len),('label_len',seq_len//2),('pred_len',pred_len),\n                ('d_feature',c_in),('d_model',d_model),('T2V_outmodel',T2V_outmodel),\n                ('d_mark',d_mark),('patch_len',patch_len),('stride',stride),('dropout',dropout)]:\n        p.add_argument(f'--{k}',default=v,type=type(v))\n    p.add_argument('--mask_spectrum',default=mask_spectrum)\n    model = D2Vformer(p.parse_args([]))\n    if not hasattr(model, 'trend_linear'):\n        model.trend_linear = nn.Linear(seq_len, d_model)\n    return model\n")
open('/content/D2Vformer/myutils/data.py','w').write("\nimport torch, pandas as pd, numpy as np, os\nfrom torch.utils.data import Dataset, DataLoader\ndef extract_time_features(df):\n    d=pd.to_datetime(df['date'])\n    return np.stack([d.dt.hour/23-.5, d.dt.weekday/6-.5,\n                     (d.dt.day-1)/30-.5, (d.dt.month-1)/11-.5],1).astype(np.float32)\nclass TSD(Dataset):\n    def __init__(self,data,stamp,seq_len,pred_len):\n        self.data,self.stamp,self.sl,self.pl=data,stamp,seq_len,pred_len\n    def __len__(self): return max(0,len(self.data)-self.sl-self.pl+1)\n    def __getitem__(self,i):\n        x=self.data[i:i+self.sl]; y=self.data[i+self.sl:i+self.sl+self.pl]\n        return (torch.tensor(x,dtype=torch.float32),torch.tensor(y,dtype=torch.float32),\n                torch.tensor(self.stamp[i:i+self.sl],dtype=torch.float32),\n                torch.tensor(self.stamp[i+self.sl:i+self.sl+self.pl],dtype=torch.float32))\ndef get_data_loaders(name,seq_len=96,pred_len=48,batch_size=64,root='/content/D2Vformer/datasets'):\n    paths={'etth1':('ETT-small','ETTh1.csv'),'exchange':('exchange_rate','exchange_rate.csv'),\n           'exchange_rate':('exchange_rate','exchange_rate.csv')}\n    folder,fname=paths[name.lower()]\n    df=pd.read_csv(os.path.join(root,folder,fname))\n    feat=[c for c in df.columns if c!='date']\n    data=df[feat].values.astype(np.float32); stamp=extract_time_features(df)\n    n=len(data); nt,nv=int(n*.6),int(n*.2)\n    tr,va,te=data[:nt],data[nt:nt+nv],data[nt+nv:]\n    trs,vas,tes=stamp[:nt],stamp[nt:nt+nv],stamp[nt+nv:]\n    mu=tr.mean(0,keepdims=True); sd=tr.std(0,keepdims=True); sd[sd==0]=1\n    tr,va,te=(tr-mu)/sd,(va-mu)/sd,(te-mu)/sd\n    mk=dict(seq_len=seq_len,pred_len=pred_len)\n    trl=DataLoader(TSD(tr,trs,**mk),batch_size,shuffle=True,drop_last=True)\n    val=DataLoader(TSD(va,vas,**mk),batch_size,shuffle=False)\n    tel=DataLoader(TSD(te,tes,**mk),batch_size,shuffle=False)\n    meta={'num_variables':data.shape[1],'train_samples':len(TSD(tr,trs,**mk)),\n          'val_samples':len(TSD(va,vas,**mk)),'test_samples':len(TSD(te,tes,**mk))}\n    return trl,val,tel,meta\n")
open('/content/D2Vformer/myutils/metrics.py','w').write('\nimport torch, numpy as np, math\ndef compute_batch_attention_entropy(A,eps=1e-12):\n    L=A.shape[-1]; e=-torch.sum(A*torch.log(A+eps),dim=-1).mean().item()\n    return e,(e/math.log(L)) if L>1 else 0.0\n')
open('/content/D2Vformer/myutils/reproducibility.py','w').write("\nimport torch,numpy as np,random,hashlib,os\ndef set_seed(seed=42):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False\n    os.environ['PYTHONHASHSEED']=str(seed)\ndef compute_parameter_checksum(model):\n    h=hashlib.sha256()\n    for n,p in sorted(model.named_parameters()):\n        h.update(n.encode()); h.update(p.detach().cpu().numpy().tobytes())\n    return h.hexdigest()\n")
# Blank __init__.py so imports work
for d in ['/content/D2Vformer/models','/content/D2Vformer/baselines','/content/D2Vformer/myutils']:
    open(f'{d}/__init__.py','w').close()
print('All source files written.')

# Import and quick sanity check
import sys; sys.path.insert(0,'/content/D2Vformer')
import torch
from models.pure_d2vformer import PureD2Vformer
from baselines.dlinear import DLinear
from baselines.persistence import PersistenceBaseline
from baselines.repository_d2vformer import create_repository_d2vformer
from myutils.data import get_data_loaders
from myutils.metrics import compute_batch_attention_entropy
from myutils.reproducibility import set_seed, compute_parameter_checksum
m=PureD2Vformer(c_in=7,seq_len=96)
x=torch.randn(2,96,7); xm=torch.randn(2,96,4); ym=torch.randn(2,48,4)
out,A=m(x,xm,ym)
assert out.shape==(2,48,7) and A.shape==(2,128,48,96)
params=sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'PureD2Vformer OK. Params={params:,}  (should be 44,021 for c_in=7,seq_len=96)')
del m,x,xm,ym,out,A
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print(f'Ready. Device={DEVICE}')


## Cell 3 — Training & evaluation helper functions

In [ ]:
import torch, torch.nn as nn, numpy as np, pandas as pd
import json, time, math, os
from tqdm.auto import tqdm

# ---- Memory-safe batch size cap ----
def safe_bs(req, O, H=128, L=96, max_bytes=256*1024*1024):
    '''Keep attention [B,H,O,L] under max_bytes (default 256 MB).'''
    return min(req, max(1, max_bytes // (H*O*L*4)))

def stream(p, t, sq, ab, n):
    d=p-t; return sq+float(np.sum(d**2)), ab+float(np.sum(np.abs(d))), n+d.size

# ─── PureD2Vformer training ───────────────────────────────────────────────────
def train_pure(ds, seq_len=96, O_train=48, d_model=128, d_ff=256, k_freq=16,
               dropout=0.05, lr=1e-3, epochs=10, patience=3, bs=64,
               seed=42, device='cpu', ckpt_dir='/content/D2Vformer/results/checkpoints'):
    set_seed(seed)
    trl,val,_,meta = get_data_loaders(ds, seq_len, O_train, bs)
    c_in = meta['num_variables']
    model = PureD2Vformer(c_in=c_in,seq_len=seq_len,d_model=d_model,
                           d_ff=d_ff,k_freq=k_freq,dropout=dropout).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt=torch.optim.Adam(model.parameters(),lr=lr); crit=nn.MSELoss()
    best,pat,ep_best=float('inf'),0,0
    ckpt=os.path.join(ckpt_dir,f'pured2v_{ds}_s{seed}.pt')
    t0=time.time()
    for ep in range(1,epochs+1):
        model.train(); tl=0; nb=0
        for bx,by,bxm,bym in tqdm(trl,desc=f'PureD2V|{ds}|s{seed}|ep{ep}',leave=False):
            bx,by,bxm,bym=bx.to(device),by.to(device),bxm.to(device),bym.to(device)
            opt.zero_grad(); out,_=model(bx,bxm,bym); loss=crit(out,by); loss.backward(); opt.step()
            tl+=loss.item(); nb+=1
        tl/=max(1,nb)
        model.eval(); vl=0; nv=0
        with torch.no_grad():
            for bx,by,bxm,bym in val:
                bx,by,bxm,bym=bx.to(device),by.to(device),bxm.to(device),bym.to(device)
                vl+=crit(model(bx,bxm,bym)[0],by).item(); nv+=1
        vl/=max(1,nv)
        print(f'  [PureD2V|{ds}|s{seed}] ep{ep}/{epochs}  train={tl:.4f}  val={vl:.4f}')
        if vl<best:
            best,pat,ep_best=vl,0,ep
            cs=compute_parameter_checksum(model)
            torch.save({'model_state_dict':model.state_dict(),'checksum':cs,
                        'param_count':n_params,'c_in':c_in,'seq_len':seq_len,'d_model':d_model,
                        'd_ff':d_ff,'k_freq':k_freq,'dropout':dropout,'train_horizon':O_train,
                        'seed':seed,'dataset':ds,'best_epoch':ep_best,'val_loss':best},ckpt)
        else:
            pat+=1
            if pat>=patience: print(f'  Early stop ep{ep}. Best val={best:.4f}@ep{ep_best}'); break
    return ckpt, time.time()-t0, n_params

# ─── DLinear training ─────────────────────────────────────────────────────────
def train_dl(ds,seq_len=96,O=48,lr=1e-3,epochs=10,patience=3,bs=64,
             seed=42,device='cpu',ckpt_dir='/content/D2Vformer/results/checkpoints'):
    set_seed(seed)
    trl,val,_,meta=get_data_loaders(ds,seq_len,O,bs); c_in=meta['num_variables']
    model=DLinear(seq_len,O,c_in).to(device)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt=torch.optim.Adam(model.parameters(),lr=lr); crit=nn.MSELoss()
    best,pat=float('inf'),0; ckpt=os.path.join(ckpt_dir,f'dlinear_{ds}_O{O}_s{seed}.pt')
    t0=time.time()
    for ep in range(1,epochs+1):
        model.train()
        for bx,by,_,_ in trl:
            bx,by=bx.to(device),by.to(device); opt.zero_grad()
            crit(model(bx),by).backward(); opt.step()
        model.eval(); vl=0; nv=0
        with torch.no_grad():
            for bx,by,_,_ in val:
                vl+=crit(model(bx.to(device)),by.to(device)).item(); nv+=1
        vl/=max(1,nv)
        if vl<best:
            best,pat=vl,0
            torch.save({'model_state_dict':model.state_dict(),'param_count':n_params,
                        'c_in':c_in,'seq_len':seq_len,'pred_len':O,'seed':seed,'dataset':ds},ckpt)
        else:
            pat+=1
            if pat>=patience: break
    return ckpt, time.time()-t0, n_params

# ─── Repository-D2Vformer training ───────────────────────────────────────────
def mask_spectrum(trl,alpha=0.2,device='cpu'):
    amps=0
    for bx,_,_,_ in trl: amps+=torch.abs(torch.fft.rfft(bx.to(device),dim=1)).mean(0).mean(1)
    return amps.topk(max(1,int(amps.shape[0]*alpha))).indices

def train_repo(ds,seq_len=96,O=48,lr=1e-3,epochs=10,patience=3,bs=64,
               seed=42,device='cpu',ckpt_dir='/content/D2Vformer/results/checkpoints'):
    set_seed(seed)
    trl,val,_,meta=get_data_loaders(ds,seq_len,O,bs); c_in=meta['num_variables']
    ms=mask_spectrum(trl,device=device)
    model=create_repository_d2vformer(c_in,seq_len,O,ms).to(device)
    if not hasattr(model, 'trend_linear'):
        model.trend_linear = nn.Linear(seq_len, model.d_model).to(device)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt=torch.optim.Adam(model.parameters(),lr=lr); crit=nn.MSELoss()
    best,pat=float('inf'),0; ckpt=os.path.join(ckpt_dir,f'repo_{ds}_O{O}_s{seed}.pt')
    t0=time.time()
    for ep in range(1,epochs+1):
        model.train()
        for bx,by,bxm,bym in trl:
            bx,by,bxm,bym=bx.to(device),by.to(device),bxm.to(device),bym.to(device)
            opt.zero_grad(); crit(model(bx,bxm,by,bym),by).backward(); opt.step()
        model.eval(); vl=0; nv=0
        with torch.no_grad():
            for bx,by,bxm,bym in val:
                bx,by,bxm,bym=bx.to(device),by.to(device),bxm.to(device),bym.to(device)
                vl+=crit(model(bx,bxm,by,bym),by).item(); nv+=1
        vl/=max(1,nv)
        if vl<best:
            best,pat=vl,0
            torch.save({'model_state_dict':model.state_dict(),'param_count':n_params,
                        'c_in':c_in,'seq_len':seq_len,'pred_len':O,'seed':seed,'dataset':ds},ckpt)
        else:
            pat+=1
            if pat>=patience: break
    return ckpt, time.time()-t0, n_params

# ─── Evaluation (all streaming/memory-safe) ───────────────────────────────────
def eval_pure_zeroshot(ckpt_path, O, ds, bs=64, device='cpu'):
    ck=torch.load(ckpt_path,map_location=device)
    model=PureD2Vformer(c_in=ck['c_in'],seq_len=ck['seq_len'],d_model=ck['d_model'],
                         d_ff=ck['d_ff'],k_freq=ck['k_freq'],dropout=ck['dropout']).to(device)
    model.load_state_dict(ck['model_state_dict']); model.eval()
    h0=ck['checksum']; assert compute_parameter_checksum(model)==h0,'Hash mismatch!'
    b=safe_bs(bs,O,ck['d_model'],ck['seq_len'])
    _,_,tel,_=get_data_loaders(ds,ck['seq_len'],O,b)
    sq,ab,n=0.,0.,0; ents,nents=[],[]
    t0=time.time()
    with torch.no_grad():
        for bx,by,bxm,bym in tel:
            bx,bxm,bym=bx.to(device),bxm.to(device),bym.to(device)
            out,A=model(bx,bxm,bym)
            sq,ab,n=stream(out.cpu().numpy(),by.numpy(),sq,ab,n)
            e,ne=compute_batch_attention_entropy(A); ents.append(e); nents.append(ne); del A,out
    assert compute_parameter_checksum(model)==h0,'Hash mutated!'
    return {'mse':sq/n,'mae':ab/n,'attention_entropy':float(np.mean(ents)),
            'normalized_attention_entropy':float(np.mean(nents)),
            'parameter_count':ck['param_count'],'inference_time':time.time()-t0,'checksum_verified':True}

def eval_dl(ckpt_path, O, ds, bs=64, device='cpu'):
    ck=torch.load(ckpt_path,map_location=device)
    model=DLinear(ck['seq_len'],ck['pred_len'],ck['c_in']).to(device)
    model.load_state_dict(ck['model_state_dict']); model.eval()
    _,_,tel,_=get_data_loaders(ds,ck['seq_len'],O,bs)
    sq,ab,n=0.,0.,0; t0=time.time()
    with torch.no_grad():
        for bx,by,_,_ in tel: sq,ab,n=stream(model(bx.to(device)).cpu().numpy(),by.numpy(),sq,ab,n)
    return {'mse':sq/n,'mae':ab/n,'attention_entropy':None,'normalized_attention_entropy':None,
            'parameter_count':ck['param_count'],'inference_time':time.time()-t0,'checksum_verified':True}

def eval_repo(ckpt_path, O, ds, bs=64, device='cpu'):
    ck=torch.load(ckpt_path,map_location=device)
    _,_,tel,_=get_data_loaders(ds,ck['seq_len'],O,bs)
    model=create_repository_d2vformer(ck['c_in'],ck['seq_len'],ck['pred_len'],torch.tensor([0,1])).to(device)
    if not hasattr(model, 'trend_linear'):
        model.trend_linear = nn.Linear(ck['seq_len'], model.d_model).to(device)
    model.load_state_dict(ck['model_state_dict']); model.eval()
    sq,ab,n=0.,0.,0; t0=time.time()
    with torch.no_grad():
        for bx,by,bxm,bym in tel:
            bx,bxm,bym=bx.to(device),bxm.to(device),bym.to(device)
            sq,ab,n=stream(model(bx,bxm,by.to(device),bym).cpu().numpy(),by.numpy(),sq,ab,n)
    return {'mse':sq/n,'mae':ab/n,'attention_entropy':None,'normalized_attention_entropy':None,
            'parameter_count':ck['param_count'],'inference_time':time.time()-t0,'checksum_verified':True}

def eval_persist(ds, O, seq_len=96, bs=64):
    _,_,tel,_=get_data_loaders(ds,seq_len,O,bs); model=PersistenceBaseline()
    sq,ab,n=0.,0.,0; t0=time.time()
    for bx,by,_,_ in tel: sq,ab,n=stream(model(bx,pred_len=O).numpy(),by.numpy(),sq,ab,n)
    return {'mse':sq/n,'mae':ab/n,'attention_entropy':None,'normalized_attention_entropy':None,
            'parameter_count':0,'inference_time':time.time()-t0,'checksum_verified':True}

print('All training and evaluation functions ready.')


## Cell 4 — Run Full Experiment

⏱️ **Expected time on T4 GPU: ~8–15 minutes**

Results are saved to `/content/D2Vformer/results/raw/results.csv` after every dataset+seed block.

In [ ]:
# ── SAFEGUARD PATCH FOR REPOSITORY-D2VFORMER ─────────────────────────────────
import torch.nn as nn
try:
    from model.D2Vformer import D2Vformer
    _orig_fwd = D2Vformer.forward
    def _safe_fwd(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        if not hasattr(self, 'trend_linear'):
            self.trend_linear = nn.Linear(self.seq_len, self.d_model).to(x_enc.device)
        return _orig_fwd(self, x_enc, x_mark_enc, x_dec, x_mark_dec)
    D2Vformer.forward = _safe_fwd

    _orig_init = D2Vformer.__init__
    def _safe_init(self, configs):
        _orig_init(self, configs)
        if not hasattr(self, 'trend_linear'):
            self.trend_linear = nn.Linear(self.seq_len, self.d_model)
    D2Vformer.__init__ = _safe_init
    print('✓ In-memory trend_linear patch applied to D2Vformer.')
except Exception as e:
    print(f'Patch notice: {e}')

# ── CONFIGURATION ──────────────────────────────────────────────────────────────
DATASETS      = ['ETTh1', 'exchange']
SEEDS         = [42, 43, 44]
TRAIN_HORIZON = 48
EVAL_HORIZONS = [24, 48, 96, 192, 336, 720]
EPOCHS, PATIENCE, BS, LR, SEQ_LEN = 10, 3, 64, 1e-3, 96
CKPT_DIR = '/content/D2Vformer/results/checkpoints'
RAW_DIR  = '/content/D2Vformer/results/raw'
CSV_PATH = f'{RAW_DIR}/results.csv'
JSON_PATH= f'{RAW_DIR}/results.json'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

all_results = []
exp_t0 = time.time()
print('='*80)
print('D2VFORMER HORIZON GENERALIZATION — FULL EXPERIMENT')
print(f'Device: {DEVICE} | Datasets: {DATASETS} | Seeds: {SEEDS}')
print(f'Train horizon: O={TRAIN_HORIZON} | Eval: {EVAL_HORIZONS}')
print('='*80)

for dataset in DATASETS:
    for seed in SEEDS:
        print(f'\n>>> Dataset: {dataset} | Seed: {seed} <<<')
        set_seed(seed)

        # 1. PureD2Vformer ─ train once, eval zero-shot across all horizons
        print(f'\n[1/4] PureD2Vformer: training at O_train={TRAIN_HORIZON}...')
        ck_p,tt_p,pc_p = train_pure(dataset,SEQ_LEN,TRAIN_HORIZON,epochs=EPOCHS,
                                     patience=PATIENCE,bs=BS,lr=LR,seed=seed,
                                     device=DEVICE,ckpt_dir=CKPT_DIR)
        print(f'  Done in {tt_p:.1f}s | params={pc_p:,}')
        for O in EVAL_HORIZONS:
            m=eval_pure_zeroshot(ck_p,O,dataset,BS,DEVICE)
            all_results.append({'dataset':dataset,'model':'PureD2Vformer',
                'regime':f'Zero-Shot(O_train={TRAIN_HORIZON})','seed':seed,
                'train_horizon':TRAIN_HORIZON,'eval_horizon':O,
                'mse':round(m['mse'],5),'mae':round(m['mae'],5),
                'attention_entropy':round(m['attention_entropy'],4),
                'normalized_attention_entropy':round(m['normalized_attention_entropy'],4),
                'parameter_count':pc_p,'training_time_sec':round(tt_p,2),
                'inference_time_sec':round(m['inference_time'],3),'checksum_verified':True})
            print(f'  ZeroShot O={O:3d} | MSE={m["mse"]:.4f} MAE={m["mae"]:.4f} H_norm={m["normalized_attention_entropy"]:.4f}')

        # 2. DLinear ─ retrained per horizon
        print(f'\n[2/4] DLinear: training 6 horizons...')
        for O in EVAL_HORIZONS:
            ck_d,tt_d,pc_d=train_dl(dataset,SEQ_LEN,O,LR,EPOCHS,PATIENCE,BS,seed,DEVICE,CKPT_DIR)
            m=eval_dl(ck_d,O,dataset,BS,DEVICE)
            all_results.append({'dataset':dataset,'model':'DLinear',
                'regime':'Horizon-Specific Retrained','seed':seed,
                'train_horizon':O,'eval_horizon':O,'mse':round(m['mse'],5),'mae':round(m['mae'],5),
                'attention_entropy':None,'normalized_attention_entropy':None,'parameter_count':pc_d,
                'training_time_sec':round(tt_d,2),'inference_time_sec':round(m['inference_time'],3),
                'checksum_verified':True})
            print(f'  DLinear   O={O:3d} | MSE={m["mse"]:.4f} MAE={m["mae"]:.4f}')

        # 3. Repository-D2Vformer ─ retrained per horizon
        print(f'\n[3/4] Repository-D2Vformer: training 6 horizons...')
        for O in EVAL_HORIZONS:
            ck_r,tt_r,pc_r=train_repo(dataset,SEQ_LEN,O,LR,EPOCHS,PATIENCE,BS,seed,DEVICE,CKPT_DIR)
            m=eval_repo(ck_r,O,dataset,BS,DEVICE)
            all_results.append({'dataset':dataset,'model':'Repository-D2Vformer',
                'regime':'Horizon-Specific Retrained','seed':seed,
                'train_horizon':O,'eval_horizon':O,'mse':round(m['mse'],5),'mae':round(m['mae'],5),
                'attention_entropy':None,'normalized_attention_entropy':None,'parameter_count':pc_r,
                'training_time_sec':round(tt_r,2),'inference_time_sec':round(m['inference_time'],3),
                'checksum_verified':True})
            print(f'  RepoD2V   O={O:3d} | MSE={m["mse"]:.4f} MAE={m["mae"]:.4f}')

        # 4. Persistence baseline
        print(f'\n[4/4] Persistence baseline...')
        for O in EVAL_HORIZONS:
            m=eval_persist(dataset,O,SEQ_LEN,BS)
            all_results.append({'dataset':dataset,'model':'Persistence',
                'regime':'Non-Learning Control','seed':seed,'train_horizon':None,'eval_horizon':O,
                'mse':round(m['mse'],5),'mae':round(m['mae'],5),'attention_entropy':None,
                'normalized_attention_entropy':None,'parameter_count':0,'training_time_sec':0.0,
                'inference_time_sec':round(m['inference_time'],3),'checksum_verified':True})
            print(f'  Persist   O={O:3d} | MSE={m["mse"]:.4f} MAE={m["mae"]:.4f}')

        # Save after each dataset+seed block
        df_res=pd.DataFrame(all_results)
        df_res.to_csv(CSV_PATH,index=False)
        with open(JSON_PATH,'w') as f: json.dump(all_results,f,indent=2)
        print(f'\n  ✓ Saved {len(all_results)} rows → {CSV_PATH}')

total=time.time()-exp_t0
print(f'\n{"="*80}')
print(f'EXPERIMENT COMPLETE in {total/60:.1f} min | {len(all_results)} total result rows')
print(f'Results: {CSV_PATH}')
print('='*80)


## Cell 5 — Publication-Quality Plots

In [ ]:
import matplotlib.pyplot as plt
try: plt.style.use('seaborn-v0_8-whitegrid')
except: pass
plt.rcParams.update({'font.size':11,'axes.labelsize':12,'axes.titlesize':13,'legend.fontsize':9})

PLOTS = '/content/D2Vformer/results/plots'
os.makedirs(PLOTS,exist_ok=True)
df=pd.read_csv(CSV_PATH)

COLORS  = {'PureD2Vformer':'#1f77b4','DLinear':'#2ca02c',
            'Repository-D2Vformer':'#d62728','Persistence':'#7f7f7f'}
MARKERS = {'PureD2Vformer':'o','DLinear':'s','Repository-D2Vformer':'^','Persistence':'x'}
LABELS  = {'PureD2Vformer':f'PureD2Vformer (Zero-Shot, O_train={TRAIN_HORIZON})',
            'DLinear':'DLinear (Retrained per horizon)',
            'Repository-D2Vformer':'Repository-D2Vformer (Retrained per horizon)',
            'Persistence':'Persistence (no learning)'}

for ds in df['dataset'].unique():
    sub=df[df['dataset']==ds]

    for metric in ['mse','mae']:
        fig,ax=plt.subplots(figsize=(9,5))
        for model in sub['model'].unique():
            g=sub[sub['model']==model].groupby('eval_horizon')[metric].agg(['mean','std']).reset_index()
            ax.plot(g['eval_horizon'],g['mean'],marker=MARKERS[model],label=LABELS[model],
                    color=COLORS[model],linewidth=2,markersize=7)
            if g['std'].notna().any():
                ax.fill_between(g['eval_horizon'],g['mean']-g['std'],g['mean']+g['std'],
                                alpha=0.12,color=COLORS[model])
        ax.set_title(f'{ds} — Test {metric.upper()} vs Prediction Horizon')
        ax.set_xlabel('Prediction Horizon (O)'); ax.set_ylabel(f'Test {metric.upper()}')
        ax.set_xticks(EVAL_HORIZONS); ax.legend(frameon=True); plt.tight_layout()
        plt.savefig(f'{PLOTS}/{ds}_{metric}_vs_horizon.png',dpi=300); plt.show()

    p=sub[sub['model']=='PureD2Vformer'].groupby('eval_horizon')['normalized_attention_entropy'].agg(['mean','std']).reset_index()
    if not p.empty:
        fig,ax=plt.subplots(figsize=(9,4.5))
        ax.plot(p['eval_horizon'],p['mean'],'o-',color='#9467bd',lw=2,ms=8,label='H_norm(A)')
        if p['std'].notna().any():
            ax.fill_between(p['eval_horizon'],p['mean']-p['std'],p['mean']+p['std'],alpha=0.15,color='#9467bd')
        ax.axhline(1.0,color='red',ls='--',alpha=0.6,label='Uniform attention (max)')
        ax.axvline(TRAIN_HORIZON,color='blue',ls=':',alpha=0.6,label=f'Training horizon O={TRAIN_HORIZON}')
        ax.set_title(f'{ds} — PureD2Vformer Cross-Temporal Attention Entropy vs Horizon')
        ax.set_xlabel('Prediction Horizon (O)'); ax.set_ylabel('Normalized Attention Entropy H(A)/log(L)')
        ax.set_xticks(EVAL_HORIZONS); ax.set_ylim(0,1.1); ax.legend(); plt.tight_layout()
        plt.savefig(f'{PLOTS}/{ds}_attention_entropy.png',dpi=300); plt.show()

# Parameter count plot
fig,ax=plt.subplots(figsize=(9,4.5))
ds0=df['dataset'].unique()[0]
for model in df[df['dataset']==ds0]['model'].unique():
    if model=='Persistence': continue
    g=df[(df['dataset']==ds0)&(df['model']==model)].drop_duplicates('eval_horizon')
    ax.plot(g['eval_horizon'],g['parameter_count'],marker=MARKERS[model],
            label=LABELS[model],color=COLORS[model],lw=2,ms=7)
ax.set_title('Trainable Parameter Count vs Prediction Horizon')
ax.set_xlabel('Prediction Horizon (O)'); ax.set_ylabel('Parameter Count')
ax.set_xticks(EVAL_HORIZONS); ax.legend(frameon=True); plt.tight_layout()
plt.savefig(f'{PLOTS}/parameter_count.png',dpi=300); plt.show()
print(f'All plots saved to {PLOTS}')


## Cell 6 — Results Summary Table

In [ ]:
df=pd.read_csv(CSV_PATH)
print('\n=== MSE/MAE SUMMARY (mean ± std across 3 seeds) ===')
summary=(df.groupby(['dataset','model','eval_horizon'])[['mse','mae']]
           .agg(['mean','std']).round(5))
print(summary.to_string())

print('\n=== NORMALIZED ATTENTION ENTROPY — PureD2Vformer ===')
ent=(df[df['model']=='PureD2Vformer']
     .groupby(['dataset','eval_horizon'])['normalized_attention_entropy']
     .agg(['mean','std']).round(4))
print(ent.to_string())

print('\n=== CUMULATIVE TRAINING TIME (avg across seeds, all 6 horizons) ===')
t=(df.groupby(['dataset','model','seed'])['training_time_sec'].sum()
     .groupby(['dataset','model']).mean().round(1))
print(t.to_string())


## Cell 7 — Save to Google Drive (optional)

Uncomment and run to persist all results and plots across Colab sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('/content/D2Vformer/results',
#                 '/content/drive/MyDrive/D2Vformer_results', dirs_exist_ok=True)
# print('Saved to Google Drive → MyDrive/D2Vformer_results/')
print('Uncomment the lines above to save to Google Drive.')
